In [2]:
#imports
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
#load env
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

In [4]:
#initialize openai

gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=google_api_key)

ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [5]:
system_prompt = """
You are a helpful assistant which helps to solve the user's problem and try to give the best solution.
Also Your reply should be well organised in Markdown.
"""

In [6]:
def stream_ollama(text):
    stream = ollama.chat.completions.create(
        model="gemma3",
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":text}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [7]:
def stream_gemini(text):
    stream = gemini.chat.completions.create(
        model = "gemini-2.5-flash-lite",
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":text}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [8]:
def stream_choose(model,text):
    if model == "Gemini":
        result = stream_gemini(text)
    elif model == "Ollama":
        result = stream_ollama(text)
    else:
        raise ValueError("Unknown Model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:",info="Enter a message for LLM",lines=7)
model_selector = gr.Dropdown(["Gemini","Ollama"],label= "Select Model",value="Gemini")
message_output = gr.Markdown(label="Response")

view = gr.Interface(
    fn=stream_choose,
    inputs=[model_selector,message_input],
    outputs=[message_output],
    examples=[
        ["Explain recursion like I'm 5 years old","Gemini"],
        ["What are the pros and cons of using Python vs JavaScript for backend development?","Ollama"]
    ],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_